# Aprendizado de Máquina — Lista prática 08

## Métricas para Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Marta trabalha na área de relacionamento de um banco. O banco quer prever quais
clientes farão uma certa transação, que ele não revela, e a equipe de Marta vai entrar
em contato com os clientes que um modelo apontar. Os dados são os do
`bank_train_redux.csv`, os mesmos do laboratório: 200 covariáveis anônimas e a
resposta $Y$, que vale 1 para quem fez a transação, cerca de 10% dos clientes. A lista
usa outra divisão entre treino e teste, e por isso os números diferem um pouco dos do
laboratório.

Ao longo da lista, $\widehat p(\mathbf{x})$ é a probabilidade de $Y=1$ estimada pela
regressão logística, e o cliente é declarado positivo quando
$\widehat p(\mathbf{x}) \ge p_0$, para um corte $p_0$ escolhido. Os exercícios seguem a
ordem do laboratório: a acurácia, a matriz de confusão, a escolha do corte e a curva
ROC.

Cada lacuna está marcada com `...`. Substitua cada uma pela sua resposta e rode a
célula.

---
## 1. Importando os pacotes

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — a acurácia e o classificador trivial

Quem ajustou o modelo informou a Marta que ele acerta 91,55% dos clientes de teste.
Antes de repassar o número, ela quer saber quanto dele se deve ao modelo, e para isso
o compara com o classificador trivial, que declara todo cliente negativo sem olhar
covariável nenhuma.

A célula a seguir lê os dados, com a mesma limpeza do laboratório, e separa 30% dos
clientes para teste. (a) Preencha o argumento que faz a divisão preservar, no treino e
no teste, a proporção de positivos.

In [ ]:
_nome = "bank_train_redux.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

banco = pd.read_csv(_caminho, nrows=40_000)
banco = banco.replace(to_replace=";", value="", regex=True)
banco = banco.rename(columns={"var_199;;;;;;;": "var_199"})
banco["var_199"] = pd.to_numeric(banco["var_199"])

Xb = banco.drop(columns=["ID_code", "target"]).astype(float).values
yb = banco["target"].values

X_tr, X_te, y_tr, y_te = skm.train_test_split(
    Xb, yb, test_size=0.3, random_state=2026, stratify=yb)      # (a)

print(f"n = {Xb.shape[0]}, d = {Xb.shape[1]}, prevalencia = {yb.mean():.4f}")
print(f"treino: {len(y_tr)} clientes, {int(y_tr.sum())} positivos")
print(f"teste : {len(y_te)} clientes, {int(y_te.sum())} positivos")

Deve imprimir `n = 40000, d = 200, prevalencia = 0.0985`,
`treino: 28000 clientes, 2759 positivos` e `teste : 12000 clientes, 1182 positivos`.
Com `stratify`, a proporção de positivos é 0,0985 nos dois conjuntos, a mesma da base.

O modelo é o do laboratório: a regressão logística, com as covariáveis padronizadas
pelas médias e pelos desvios-padrão do treino, as duas etapas encadeadas num
`Pipeline`. O método `predict_proba` devolve uma coluna por classe, na ordem de
`modelo.classes_`, que aqui é `[0, 1]`. (b) Extraia $\widehat p$, a probabilidade
estimada de $Y=1$. (c) Calcule a acurácia do classificador trivial.

In [ ]:
modelo = Pipeline([("sc", StandardScaler()),
                   ("lg", skl.LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)
p = modelo.predict_proba(X_te)[:, 1]                            # (b)
pred = (p >= 0.5).astype(int)

acuracia = np.mean(pred == y_te)
trivial = np.mean(y_te == 0)                                    # (c)
print(f"acuracia do modelo, corte 0,5   : {acuracia:.4f}")
print(f"acuracia de 'ninguem e positivo': {trivial:.4f}")
print(f"diferenca                       : {acuracia - trivial:+.4f}")
print(f"\nclientes declarados positivos: {pred.sum()} de {len(y_te)}")

Deve imprimir:

```
acuracia do modelo, corte 0,5   : 0.9155
acuracia de 'ninguem e positivo': 0.9015
diferenca                       : +0.0140

clientes declarados positivos: 484 de 12000
```

O classificador trivial acerta todos os negativos e erra todos os positivos; a sua
acurácia é, portanto, a fração de negativos no teste, $10\,818/12\,000 = 0{,}9015$. O
modelo fica 1,40 ponto percentual acima dela. A última linha mostra por que a
distância é tão pequena: no corte 0,5 o modelo declara positivos só 484 clientes,
menos da metade dos 1 182 que fizeram a transação, e em todos os outros toma a mesma
decisão que o trivial.

O número de 91,55% não está errado, mas não responde à pergunta de Marta. Com 90% de
negativos, uma regra que nunca diz "positivo" já acerta 90%, e a acurácia quase não
registra o que acontece com a classe rara, que é a que interessa à equipe. O próximo
exercício separa os dois tipos de erro.

---
## Exercício 2 — a matriz de confusão e as taxas

Para ver o que o modelo faz em cada classe, Marta separa os acertos e os erros do
corte 0,5 em quatro contagens: os verdadeiros positivos (VP), os falsos positivos
(FP), os falsos negativos (FN) e os verdadeiros negativos (VN). A função
`confusion_matrix` devolve essas contagens com a classe real nas linhas e a prevista
nas colunas, e o `ravel` as lê linha a linha, na ordem VN, FP, FN, VP. Na aula, a
orientação é a transposta: a classe prevista, $\widehat Y$, nas linhas, e a real, $Y$,
nas colunas. (a) Monte a tabela na orientação da aula.

In [ ]:
mc = confusion_matrix(y_te, pred)          # o scikit-learn devolve o real nas linhas
vn, fp, fn, vp = mc.ravel()
P, N = vp + fn, fp + vn
n = len(y_te)

tabela = pd.DataFrame(mc.T, index=["previsto 0", "previsto 1"],     # (a)
                      columns=["real 0", "real 1"])
print(tabela)
print(f"\nP = {P} positivos e N = {N} negativos, n = {n}")

Deve imprimir

```
            real 0  real 1
previsto 0   10660     856
previsto 1     158     326
```

e `P = 1182 positivos e N = 10818 negativos, n = 12000`. Os totais das colunas são os
das classes reais, $P = VP + FN$ e $N = FP + VN$. O da segunda linha é o das decisões
positivas, $VP + FP = 326 + 158 = 484$, os clientes declarados positivos no
Exercício 1.

Cada taxa divide uma contagem pelo total de uma coluna ou da tabela inteira. Como o
teste não foi usado no ajuste, cada uma estima uma probabilidade sobre um cliente novo.
A sensibilidade, $VP/P$, estima $\mathbb{P}(\widehat Y=1\mid Y=1)$; a taxa de falsos
positivos, $FP/N$, estima $\mathbb{P}(\widehat Y=1\mid Y=0)$, e a especificidade,
$VN/N$, é o seu complemento; e a taxa de erro, $(FP+FN)/n$, estima
$\mathbb{P}(\widehat Y\neq Y)$. (b) Complete a sensibilidade e a taxa de falsos
positivos.

In [ ]:
sens = vp / P                                                 # (b)
taxa_fp = fp / N                                              # (b)
espec = vn / N
taxa_erro = (fp + fn) / n

print(f"sensibilidade            : {sens:.4f}")
print(f"taxa de falsos positivos : {taxa_fp:.4f}")
print(f"especificidade           : {espec:.4f}")
print(f"taxa de erro             : {taxa_erro:.4f}")

Deve imprimir:

```
sensibilidade            : 0.2758
taxa de falsos positivos : 0.0146
especificidade           : 0.9854
taxa de erro             : 0.0845
```

O modelo encontra 27,6% dos clientes que fizeram a transação e acusa 1,46% dos que não
fizeram. As três primeiras taxas olham uma classe real de cada vez. A taxa de erro
mistura as duas, como a acurácia, e por isso fica perto da do classificador trivial,
que erra exatamente os 9,85% de positivos; separadas, as taxas mostram que o modelo
deixa passar mais de 70% dos clientes que fizeram a transação.

---
## Exercício 3 — o corte é uma escolha

A equipe de Marta consegue fazer 1 000 contatos por mês, e o corte 0,5 aponta só 484
clientes. Para usar a capacidade inteira, Marta ordena os clientes por $\widehat p$ e
fica com os 1 000 primeiros. Isso é escolher outro corte: $p_0$ igual ao milésimo
maior valor de $\widehat p$, de modo que exatamente 1 000 clientes tenham
$\widehat p \ge p_0$,
já que os 12 000 escores do teste são todos distintos. Para avaliar a regra, Marta a
aplica aos clientes de teste, como se fossem os clientes de um mês.

A função `taxas` refaz as contas do Exercício 2 para um corte qualquer. (a) Complete a
taxa de erro na função. (b) Calcule o corte que seleciona os 1 000 clientes de maior
$\widehat p$.

In [ ]:
def taxas(y, prob, p0):
    "Contagens e taxas da regra que declara positivo quando prob >= p0."
    vn_c, fp_c, fn_c, vp_c = confusion_matrix(y, (prob >= p0).astype(int)).ravel()
    return {"declarados": vp_c + fp_c,
            "sensibilidade": vp_c / (vp_c + fn_c),
            "taxa de falsos positivos": fp_c / (fp_c + vn_c),
            "taxa de erro": (fp_c + fn_c) / len(y)}                  # (a)


p0_mil = np.sort(p)[-1000]                                     # (b)

for nome, p0 in [("corte 0,5", 0.5), ("1 000 contatos", p0_mil)]:
    r = taxas(y_te, p, p0)
    print(f"{nome:15s} p0 = {p0:.4f}  declarados {r['declarados']:5d}  "
          f"sensibilidade {r['sensibilidade']:.4f}  FPR {r['taxa de falsos positivos']:.4f}  "
          f"erro {r['taxa de erro']:.4f}")

Deve imprimir:

```
corte 0,5       p0 = 0.5000  declarados   484  sensibilidade 0.2758  FPR 0.0146  erro 0.0845
1 000 contatos  p0 = 0.3234  declarados  1000  sensibilidade 0.4349  FPR 0.0449  erro 0.0962
```

O modelo é o mesmo nas duas linhas; só o corte mudou, e cada corte define um
classificador diferente. Com 1 000 contatos a equipe alcança 514 dos 1 182 clientes que
fizeram a transação, contra 326 no corte 0,5: a sensibilidade sobe de 0,276 para 0,435.
O preço está nos falsos positivos. Dos 516 contatos a mais, 188 alcançam quem fez a
transação e 328 não; a taxa de falsos positivos sobe de 0,0146 para 0,0449, e a taxa
de erro, de 0,0845 para 0,0962.

Mesmo assim, a segunda regra é a melhor das duas para a equipe, porque o objetivo dela
não é errar pouco, e sim encontrar, com 1 000 contatos, o maior número de clientes que
farão a transação. Suponha que a ordem de $\widehat p$ seja a mesma da probabilidade
verdadeira $\mathbb{P}(Y=1\mid\mathbf{X}=\mathbf{x})$. Então os 1 000 maiores valores
de $\widehat p$ são os dos 1 000 clientes de maior probabilidade verdadeira, e nenhuma
outra escolha de 1 000 clientes tem número esperado de positivos maior, porque esse
número é a soma das probabilidades verdadeiras dos escolhidos. Para essa decisão importa
só a ordem dos escores, e é a ordem que o Exercício 4 avalia.

O corte $p_0$ foi calculado só com os valores de $\widehat p$, sem os rótulos do teste;
por isso a sensibilidade e a taxa de falsos positivos medidas nele não foram
favorecidas pela escolha.

Para ver a troca inteira, a célula seguinte varre os cortes de 0,02 a 0,95 e desenha,
para cada um, a sensibilidade, a taxa de falsos positivos e a taxa de erro, com os dois cortes acima
marcados. Ela localiza também o corte de menor taxa de erro na grade e conta os
clientes cujo $\widehat p$ fica entre 0,5 e esse corte.

In [ ]:
cortes = np.linspace(0.02, 0.95, 200)
curvas = pd.DataFrame([taxas(y_te, p, p0) for p0 in cortes], index=cortes)
p0_erro = curvas["taxa de erro"].idxmin()
entre = (p >= 0.5) & (p < p0_erro)
print(f"menor taxa de erro da grade: {curvas['taxa de erro'].min():.4f}, no corte {p0_erro:.3f}")
print(f"clientes com p entre 0.500 e {p0_erro:.3f}: {entre.sum()}, "
      f"dos quais {y_te[entre].sum()} positivos")

fig, ax = subplots(figsize=(6.8, 3.4))
for nome in ["sensibilidade", "taxa de falsos positivos", "taxa de erro"]:
    ax.plot(cortes, curvas[nome], label=nome)
ax.axvline(0.5, ls=":", color="gray", label="corte 0,5")
ax.axvline(p0_mil, ls="--", color="gray", label="1 000 contatos")
ax.set_xlabel("corte $p_0$")
ax.set_ylabel("valor")
ax.legend(fontsize=7.5, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.tight_layout()

Deve imprimir `menor taxa de erro da grade: 0.0842, no corte 0.567` e
`clientes com p entre 0.500 e 0.567: 118, dos quais 57 positivos`.

Ao baixar o corte, a sensibilidade e a taxa de falsos positivos sobem, porque o modelo
passa a declarar positivos clientes com $\widehat p$ cada vez menor. A taxa de erro é mínima
perto de 0,5, como prevê a teoria: se $\widehat p$ estima bem
$\mathbb{P}(Y=1\mid\mathbf{X}=\mathbf{x})$, o corte que minimiza a probabilidade de
erro é $1/2$, o do classificador de Bayes (Aula 07).

O mínimo da grade não cai exatamente em 0,5 porque, perto dali, a curva é quase plana.
Subir o corte de 0,5 para 0,567 deixa de declarar positivos os 118 clientes da segunda
linha: os 57 que fizeram a transação viram falsos negativos, os 61 restantes deixam de
ser falsos positivos, e os erros caem de 1 014 para 1 010. Se $\widehat p$ estima bem a
probabilidade, é isso que se espera de clientes com $\widehat p$ pouco acima de $1/2$:
perto de metade deles faz a transação, e declará-los positivos ou não dá quase o mesmo
número de erros. O mínimo foi localizado com os rótulos do teste e serve para comparar
com a teoria, não para escolher o corte.

---
## Exercício 4 — a curva ROC e a AUC

A capacidade da equipe muda de um mês para outro, e com ela o corte. Para avaliar o
modelo sem fixar um corte, Marta usa a curva ROC, o conjunto dos pontos (taxa de falsos
positivos, sensibilidade) obtidos quando $p_0$ percorre $[0,1]$, e a AUC, a área sob ela.
(a) Calcule a AUC. A célula marca na curva os dois cortes do Exercício 3.

In [ ]:
fpr, tpr, _ = roc_curve(y_te, p)
auc = roc_auc_score(y_te, p)                                  # (a)
print(f"AUC = {auc:.4f}")

fig, ax = subplots(figsize=(5, 4))
ax.plot(fpr, tpr, lw=1.6, label="logistica")
ax.plot([0, 1], [0, 1], ls="--", lw=1, color="gray", label="sorteio")
for nome, p0, marca in [("corte 0,5", 0.5, "o"), ("1 000 contatos", p0_mil, "s")]:
    r = taxas(y_te, p, p0)
    ax.plot(r["taxa de falsos positivos"], r["sensibilidade"], marca, color="crimson",
            ms=6, label=nome)
ax.set_xlabel("taxa de falsos positivos")
ax.set_ylabel("sensibilidade")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()

Deve imprimir `AUC = 0.8529`.

Cada ponto da curva é um classificador, e os dois cortes do Exercício 3 estão nela, em
$(0{,}0146;\ 0{,}2758)$ e $(0{,}0449;\ 0{,}4349)$. Escolher o número de contatos é
escolher um ponto da curva; a curva inteira descreve o que o modelo oferece em todos
os cortes. A diagonal é o desempenho de quem sorteia, sem olhar as covariáveis:
declarar cada cliente positivo com probabilidade $q$, independentemente de tudo, faz
$\mathbb{P}(\widehat Y=1\mid Y=1) = \mathbb{P}(\widehat Y=1\mid Y=0) = q$.

Por fim, uma propriedade que importa a Marta porque a prevalência muda: num mês de
campanha, a fração de clientes que faz a transação pode ser outra. A sensibilidade e a
taxa de falsos positivos condicionam na classe verdadeira e, por isso, não dependem da
prevalência: se só a proporção de positivos mudar, mantida a distribuição das
covariáveis dentro de cada classe, o que elas estimam fica igual, e a curva ROC também.

Para ver isso nos dados, a célula a seguir forma um conjunto com prevalência de 50%: os
1 182 positivos do teste e 1 182 negativos sorteados, sem reposição, entre os 10 818.
Sortear os negativos ao acaso muda a proporção das classes sem mudar a distribuição das
covariáveis dentro de cada uma. (b) Sorteie, com `rng.choice`, os índices dos
negativos.

In [ ]:
rng = np.random.default_rng(2026)
i_pos = np.flatnonzero(y_te == 1)
i_neg = rng.choice(np.flatnonzero(y_te == 0), size=len(i_pos), replace=False)    # (b)
i_eq = np.concatenate([i_pos, i_neg])

for nome, yy, pp in [("teste inteiro", y_te, p), ("prevalencia 50%", y_te[i_eq], p[i_eq])]:
    r = taxas(yy, pp, 0.5)
    print(f"{nome:16s} prevalencia {yy.mean():.4f}  AUC {roc_auc_score(yy, pp):.4f}  "
          f"sensibilidade {r['sensibilidade']:.4f}  FPR {r['taxa de falsos positivos']:.4f}")

Deve imprimir:

```
teste inteiro    prevalencia 0.0985  AUC 0.8529  sensibilidade 0.2758  FPR 0.0146
prevalencia 50%  prevalencia 0.5000  AUC 0.8488  sensibilidade 0.2758  FPR 0.0127
```

A sensibilidade é a mesma nas duas linhas, porque os positivos são os mesmos. A taxa de
falsos positivos passa de 0,0146 para 0,0127, e a AUC, de 0,8529 para 0,8488: os 1 182
negativos sorteados são uma amostra dos 10 818, e as estimativas mudam com o sorteio,
mas estimam as mesmas quantidades, que não dependem da prevalência.

Para Marta, a consequência é prática. A curva ROC e a AUC estimadas num mês continuam
valendo num mês de prevalência diferente, desde que a distribuição das covariáveis
dentro de cada classe não mude.

> **Sua vez.** A AUC do conjunto equilibrado depende de quais negativos foram
> sorteados. Repita o sorteio com 100 sementes diferentes, guarde a AUC de cada
> conjunto e veja quanto ela varia. A diferença entre 0,8488 e 0,8529 é grande diante
> dessa variação?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | acurácia 0,9155, contra 0,9015 do classificador trivial; no corte 0,5 o modelo declara positivos 484 clientes, para 1 182 positivos |
| 2 | sensibilidade 0,276, taxa de falsos positivos 0,015 e especificidade 0,985; a taxa de erro, 0,0845, mistura as duas classes e fica perto da do trivial |
| 3 | com 1 000 contatos (corte 0,3234), a sensibilidade sobe para 0,435 e a taxa de falsos positivos para 0,045; a taxa de erro é mínima perto de 0,5, onde a curva é quase plana |
| 4 | AUC 0,8529; com prevalência de 50%, a sensibilidade não muda, e a taxa de falsos positivos e a AUC só variam com o sorteio dos negativos |

**A seguir.** A Aula 09 traz um classificador construído sobre uma ideia geométrica, a
margem, e que não estima probabilidades.